# The Volebny Kompas datasets (Mintal et al. 2024), tested on Figshare item 24949470

**Paper.** Mintal, Borseková, Cicchi, Müller, Vancel, Šimková and Deegan-Krause, "The Volebny Kompas
Datasets on Slovak Voter and Party Positions", *Scientific Data* 11, 2024,
[doi:10.1038/s41597-024-03777-0](https://doi.org/10.1038/s41597-024-03777-0).

**Dataset.** Figshare item 24949470, version 1,
[doi:10.6084/m9.figshare.24949470.v1](https://doi.org/10.6084/m9.figshare.24949470.v1), published
2024-07-09. The paper cites the item without a version (10.6084/m9.figshare.24949470); version 1 is the
only version the archive lists, so it is the most recent published version and the one pinned here. The
files are streamed from Figshare into memory; the cell below lists them with their MD5 from the
archive's file list.

**Claims.** `claims/figshare-24949470/10.1038-s41597-024-03777-0.yaml`, written from the version of
record (PMC11374779). The paper's figures 2, 3 and 7, which two claims compare with, were read from
the published version of record by a separate agent that had not seen the computed placements, by pixel
position against the figures' axes; its readings, with the pixel positions of the axes they rest on,
are typed into this notebook. The deposited R code, `Volebny_Kompas_code.R`, is in the same item and is read
in the data check; where it fixes a detail the claims file leaves open, the section says so.

| Claim | Tested here | Outcome (see the record) |
|---|---|---|
| user-responses-count | yes | |
| collection-window | yes | |
| party-positions-structure | yes | |
| eleven-parties | yes | |
| party-self-placements | yes | |
| coder-agreement-kappa | yes | |
| uni-and-landscape-placements | yes, with the paper's Figs. 2 and 3 | |
| party-clustering | yes, with the paper's Fig. 7 | |
| table2-gender, table2-age, table2-education | yes | |
| age-education-inconsistencies | yes | |
| s2-s3-wide-range, s4-s5-consensus | yes | |
| efa-eleven-factors, efa-loadings-consistent | yes | |
| selected-parties-top-vote | blocked: official results not available in a pinned form | |
| website-views | blocked: the platform's usage records are not public | |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import pdist
import vk

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
files = vk.file_list()
meta = files.attrs["meta"]
print(meta["title"], "| version", meta["version"], "|", meta["doi"], "| published", meta["published_date"])
print(files.to_string(index=False))

## Data check

Everything the item holds is loaded and listed: the README, the data dictionary and the R code (text),
the user responses (CSV, `;`-separated), every sheet of the two workbooks, the recoded party positions
and the census table. The checks of the description against the values follow the printout.

In [ ]:
texts = {n: vk.read_bytes(vk.file_id(files, n)).decode("utf-8") for n in ["README.txt", "Data Dictionary.txt", "Volebny_Kompas_code.R"]}
for n, t in texts.items():
    print(f"{n}: {len(t.splitlines())} lines")
raw, likert = vk.load_responses(files)
supp = vk.load_supplementary_table(files)
sheets, pp = vk.load_party_positions(files, supp)
rec = vk.load_recoded(files)
census = pd.read_csv(pd.io.common.BytesIO(vk.read_bytes(vk.file_id(files, "census_final.csv"))))
md5 = files.assign(streamed=files["id"].map(vk.STREAMED_MD5))
print("MD5 of the streamed bytes equals the archive's for every file:", bool((md5["streamed"] == md5["computed_md5"]).all()),
      "| files streamed:", int(md5["streamed"].notna().sum()), "of", len(md5))

print("\nUser responses:", raw.shape, "columns:", list(raw.columns))
print("Values in S1 to S39 (as read):", pd.Series(raw[vk.STATEMENTS].to_numpy().ravel()).value_counts().to_dict())
for c in ["UN2", "UN3", "UN4"]:
    v = raw[c]
    print(f"{c}: {(v == '').sum()} empty, {v[v != ''].nunique()} distinct codes, e.g. {sorted(v[v != ''].unique())[:5]}")
print("UN5 to UN26 values:", sorted(set(raw[[f'UN{i}' for i in range(5, 27)]].to_numpy().ravel())))
print("Share of 1 in UN5 to UN26:", raw[[f'UN{i}' for i in range(5, 27)]].astype(int).mean().round(3).to_dict())
for c in ["GENDER", "AGE", "EDUCATION"]:
    print(c, raw[c].value_counts().to_dict())
blank = raw[["GENDER", "AGE", "EDUCATION"]].eq("")
print("Rows with gender, age, education empty:", blank.sum().to_dict(), "| all three empty together:", int(blank.all(axis=1).sum()),
      "| some but not all empty:", int((blank.any(axis=1) & ~blank.all(axis=1)).sum()))
email = raw["EMAIL_ID"]
print("EMAIL_ID: NA", int((email == "NA").sum()), "| given", int((email != "NA").sum()), "| distinct", email[email != "NA"].nunique(),
      "| largest repeat", int(email[email != "NA"].value_counts().max()))
ts = pd.to_datetime(raw["TIME.STAMP"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
print("TIME.STAMP: unparsed", int(ts.isna().sum()), "| first", ts.min(), "| last", ts.max())
print("Rows identical in every column to another row:", int(raw.duplicated(keep=False).sum()),
      "| identical in TIME.STAMP and S1 to S39:", int(raw.duplicated(["TIME.STAMP"] + vk.STATEMENTS, keep=False).sum()))

In [ ]:
print("Workbook sheets:", {k: v.shape for k, v in sheets.items()})
cs = sheets["Codesheet"]
print("Codesheet columns:", list(cs.columns))
print("Rows naming a party:", len(pp), "| rows below them (a legend):", len(cs) - len(pp))
print(cs.loc[cs["Party"].isna(), "Unique ID"].dropna().tolist())
for c in ["Self-placement position", "Self-placement source", "Expert position I", "Expert position II", "Coders Match", "Final position", "Final source"]:
    print(f"{c}: {pp[c].value_counts(dropna=False).to_dict()}")
print("Non-empty text columns:", {c: int(pp[c].notna().sum()) for c in ["Self-placement snippet", "Self-placement link", "Expert snippet I", "Expert link I",
                                                                         "Expert snippet II", "Expert link II", "Final link", "Final snippet"]})
mism = pp[pp["statement"] != pp["Statement no."]]
print(f"\nRows whose 'Statement no.' differs from the number of their English wording in Supplementary Table 1: {len(mism)} of {len(pp)}")
print(mism.groupby(["Statement no.", "statement"]).size().rename("rows").reset_index().to_string(index=False))
print("Every row's wording matched a statement:", bool(pp["statement"].notna().all()))
print("\nSupplementary Table 1:", supp.shape, list(supp.columns))
print("Recoded party positions:", rec.shape, "values", sorted(set(rec.to_numpy().ravel())))
final = pp.pivot(index="Party", columns="statement", values="Final position").replace(vk.LIKERT).replace("No opinion", 50).astype(float)
by_no = pp.pivot(index="Party", columns="Statement no.", values="Final position").replace(vk.LIKERT).replace("No opinion", 50).astype(float)
names = {"OĽANO": "OLANO"}
r2 = rec.rename(index=names)
r2.columns = range(1, 40)
print("Recoded CSV equals the workbook's final positions (No opinion as 50), statements numbered by wording:",
      f"{(r2 == final.loc[r2.index]).to_numpy().mean():.3f}", "| numbered by 'Statement no.':", f"{(r2 == by_no.loc[r2.index]).to_numpy().mean():.3f}")
print("\nCensus table:", census.shape, list(census.columns))
print(census.drop(columns="Total_Observations").nunique().to_dict())

Checks of the description against the data:

- The MD5 of every streamed file equals the one the archive lists.
- The CSV has 134,699 rows and the 69 columns of the data dictionary, in its order. "No opinion" is the
  string `NA` in S1 to S39, not an empty string as the paper's Data cleaning says; unanswered statements
  are -1, as described. The S values are 0, 25, 50, 75 and 100 only.
- UN2 to UN4 hold the three most salient statements as codes such as `6-1` (the dictionary's two-digit
  identifiers, block and item), not as S numbers. UN5 to UN15 and UN16 to UN26 are 0/1, as described.
  Gender, age and education are empty together in the same rows; the dictionary gives no categories, and
  the values are those of the R code's validation rules. The age labels use a hyphen ("18-20") where the
  paper prints an en dash; they are the same labels.
- Table 1 of the paper shows `2023-10-09 08:57:38` as a sample timestamp: it is the first row of the file,
  which is sorted newest first, and the last date in it. The timestamps cover 5 September to 9 October 2023 (see collection-window).
- EMAIL_ID repeats, as the paper says it may. The README says each row is "a unique respondent"; the paper
  says each row is a unique response. The repeats show that some respondents answered more than once.
- The party workbook has one sheet with 429 coded rows and a legend of the categories below them. Its
  "Statement no." column disagrees with the English wording for 99 rows: the rows worded as statements 16
  to 24 are numbered 22, 23, 24, 16, 17, 18, 19, 20, 21 for every party. The recoded CSV follows the
  wording, and so do the paper's Figures 2 and 3 (see uni-and-landscape-placements), so the wording is
  taken as right and the number as wrong. All analyses here number statements by their wording.
- The data dictionary lists "Self-placement source" as a column; it is empty in all 429 rows.
- This is survey data with one timestamp per response and no trial structure, so the protocol's check that
  all streams share one clock does not apply.

## The deposited R code

The lines of `Volebny_Kompas_code.R` that the analysis relies on, printed from the streamed file: the
straight-line rows, the removal of incomplete rows, "no opinion" as 50, the validation rules for gender, age
and education, the unique-email rows, the parallel analyses and factor analyses, the clustering of the recoded
party positions and the optional moratorium filter. The README's description of the recoded party positions
follows.

In [ ]:
rlines = texts["Volebny_Kompas_code.R"].splitlines()
excerpts = [("# Identify and exclude rows where all values", 3), ("rows_with_minus_one <-", 8), ("# Replace NA with 50", 5),
            ("# Define validation rules considering NA as valid", 6), ("## FILTERING DATA WITH UNIQUE EMAIL ID", 11),
            ("cor_matrix <- cor(", 1), ("parallel_pearson <-", 1), ("fa_result_ml <-", 1), ("fa_result_pa <-", 1),
            ('data <- read.csv("recoded_party_positions.csv")', 1), ("dist_matrix <-", 1), ("hclust_result <-", 1),
            ("polychoric_matrix <- polychoric", 1), ("parallel_polychoric <-", 1), ("fa_result <- fa(polychoric_matrix", 1),
            ("## FILTER OUT DATA UNTIL ELECTION MORATORIUM", 18)]
for anchor, n in excerpts:
    i = next(k for k, l in enumerate(rlines) if l.startswith(anchor))
    for k in range(i, i + n):
        print(f"{k + 1:4d}  {rlines[k]}")
    print("     ...")
print("lines naming both AGE and EDUCATION:", [l.strip() for l in rlines if "AGE" in l and "EDUCATION" in l])
print("\nREADME on the recoded party positions:")
print("\n".join(l for l in texts["README.txt"].splitlines() if "recoded them" in l))


## Row selections

Four sets of rows are used, as the claims file names them:

- `raw`: all 134,699 rows;
- `paper`: the paper's cleaning (Technical Validation, Data cleaning): rows with -1 in any of S1 to S39 removed;
- `code`: the deposited R code's cleaning, which also removes rows that give the same answer to all 39
  statements (`same_response_rows` in `Volebny_Kompas_code.R`), a step the paper does not mention;
- `code_email`: the R code's rows for the factor analysis, `code` rows whose EMAIL_ID is given and occurs once.

In [ ]:
sel = vk.selections(raw, likert)
incomplete = (likert == -1).any(axis=1)
same = likert.fillna(-999).nunique(axis=1) == 1
print({k: int(v.sum()) for k, v in sel.items()})
print("rows with -1:", int(incomplete.sum()), "| rows with one answer to all 39 (and no -1):", int((same & ~incomplete).sum()))
print("answers of those straight-line rows:", likert[same & ~incomplete].iloc[:, 0].fillna(-999).value_counts().rename({-999: "all NA"}).to_dict())

## user-responses-count

> The User Responses Dataset holds 134,699 voter responses, each with answers to the 39 statements, along
> with demographics, the three most salient statements and the parties the respondent would and would never
> vote for, for most responses.

In [ ]:
table1 = ["TIME.STAMP"] + vk.STATEMENTS + [f"UN{i}" for i in range(2, 27)] + ["GENDER", "AGE", "EDUCATION", "EMAIL_ID"]
print("rows of the raw file:", len(raw), "| expected 134,699 +- 13,470")
print("columns of Table 1 and the dictionary present:", all(c in raw.columns for c in table1), "| missing:", [c for c in table1 if c not in raw.columns])
print("rows after removing incomplete ones (paper):", int(sel["paper"].sum()), "| after the R code's cleaning:", int(sel["code"].sum()))
share = {"demographics given": (raw["GENDER"] != "").mean(), "a salient statement given": (raw["UN2"] != "").mean(),
         "a party considered": (raw[[f'UN{i}' for i in range(5, 16)]].astype(int).sum(axis=1) > 0).mean(),
         "a party never considered": (raw[[f'UN{i}' for i in range(16, 27)]].astype(int).sum(axis=1) > 0).mean()}
print({k: round(float(v), 3) for k, v in share.items()})

**Result: reproduced.** The raw file has 134,699 rows (expected 134,699) and every column of Table 1; 81.0% of rows give demographics and 67.6% a salient statement, so "for most responses" holds. 133,473 rows remain after removing incomplete ones.

## collection-window

> The responses were gathered during the 26 days leading up to and including election day, from the launch
> on 5 September 2023 to the election on 30 September 2023.

Readings from the criterion: (a) timestamps as given; (b) timestamps read as UTC and converted to Slovak
time (Europe/Bratislava, CEST). The R code's optional filter ends at 2023-09-30 22:42:30, "the time of the last
response before the end of the election moratorium", and is shown as well.

In [ ]:
start, end = pd.Timestamp("2023-09-05"), pd.Timestamp("2023-10-01")
readings = {"(a) as given": ts, "(b) UTC to Slovak time": ts.dt.tz_localize("UTC").dt.tz_convert("Europe/Bratislava").dt.tz_localize(None)}
for name, t in readings.items():
    out = (t < start) | (t >= end)
    print(f"{name}: first {t.min()}, last {t.max()}; outside 5 to 30 Sep: {int(out.sum())} of {len(t)} ({out.mean():.2%});",
          "dates outside:", t[out].dt.date.value_counts().sort_index().to_dict())
late = ts > pd.Timestamp("2023-09-30 22:42:30")
print("rows after the R code's moratorium end (2023-09-30 22:42:30, as given): raw", int(late.sum()),
      "| data_cleaned_corrected (the R code's cleaning, to which the code applies the filter):", int((late & sel["code"]).sum()))
for name, t in readings.items():
    print(f"days with responses inside the window, {name}:", t[(t >= start) & (t < end)].dt.date.nunique())
print("first three rows of the file:", raw["TIME.STAMP"].head(3).tolist(), "| first row is the latest timestamp:", bool(ts.iloc[0] == ts.max()),
      "| file sorted newest first:", bool(ts.is_monotonic_decreasing))
daily = ts.dt.date.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.bar(pd.to_datetime(daily.index), daily.values)
ax.set_ylabel("responses per day")
ax.set_title(f"Responses per day by TIME.STAMP as given ({daily.min()} to {daily.max()} per day)")
plt.show()

**Result: partially reproduced.** Under both readings the timestamps start on 5 September 2023 and cover all 26 days up to election day, but 432 responses as given (0.32%), and 494 read as UTC (0.37%), fall from 1 to 9 October 2023, after election day, so the criterion, every timestamp inside the window, fails; 99.7% of rows are inside it. The file is sorted newest first, so its first row, the sample of Table 1 (2023-10-09 08:57:38), is the latest timestamp and one of the late rows. The R code's optional filter, ending at 2023-09-30 22:42:30, would drop 451 rows of `data_cleaned_corrected`, the rows it is applied to (461 of the raw rows). The code parses both the timestamps and the cut-off with `tz = "UTC"`, so it compares the values as given, and its comment gives the moratorium's end as 22:45 with no time zone; this suggests the timestamps are local time (reading a) but does not settle it. The outcome is the same under both readings.

## party-positions-structure

> The Party Positions Dataset lists the 39 statements in Slovak, Hungarian and English and holds 429 party
> positions, both party self-placements and expert-coded positions, with sources, snippets and links.

In [ ]:
langs = ["Statement in English", "Statement in Slovak", "Statements in Hungarian"]
per_statement = pp.groupby("statement")[langs].nunique()
print("coded rows (positions):", len(pp), "| statements:", pp["statement"].nunique(), "| by 'Statement no.':", pp["Statement no."].nunique())
print("every row has all three wordings:", bool(pp[langs].notna().all().all()),
      "| distinct wordings per statement (max):", per_statement.max().to_dict())
cols = ["Self-placement position", "Expert position I", "Expert position II", "Final position", "Self-placement source",
        "Self-placement snippet", "Self-placement link", "Expert snippet I", "Expert link I", "Final source", "Final snippet", "Final link"]
print({c: int(pp[c].notna().sum()) for c in cols})

**Result: reproduced.** 429 positions on 39 statements, each with its Slovak, Hungarian and English wording; there are self-placement, expert (two coders) and final positions, with snippet and link columns for each and a source column for the final positions. The self-placement source column is empty.

## eleven-parties

> Eleven parties are positioned on the 39 statements.

In [ ]:
cover = pp.groupby("Party")["statement"].agg(["nunique", "size"])
print(cover.to_string())
print("parties:", pp["Party"].nunique(), "| each with 39 distinct statements:", bool((cover["nunique"] == 39).all()))
print("recoded CSV parties:", rec.index.tolist())

**Result: reproduced.** The workbook names 11 parties, each with a position on all 39 statements.

## party-self-placements

> Four of the eleven parties returned the self-placement questionnaire.

In [ ]:
sp = pp.groupby("Party")["Self-placement position"].agg(lambda s: int(s.notna().sum()))
print(sp.sort_values(ascending=False).to_string())
print("parties with at least one self-placement:", int((sp > 0).sum()))
print("values used by PS:", pp.loc[pp["Party"] == "PS", "Self-placement position"].value_counts().to_dict())

**Result: reproduced.** Four parties have self-placements: SaS on 39 statements, DEMOKRATI on 38, PS on 32 and KDH on 22. PS answered only "agree" or "disagree", as the workbook's note says.

## coder-agreement-kappa

> Agreement between the two expert coders of party positions was high, with a weighted Cohen's kappa of 0.84.

Readings from the criterion: weights (a) linear or (b) quadratic; "No opinion" (i) pairs with No opinion from
either coder left out, or (ii) scored as Neutral. The expected value is 0.84 +- 0.084.

In [ ]:
cats = ["Completely disagree", "Tend to disagree", "Neutral", "Tend to agree", "Completely agree"]
a, b = pp["Expert position I"], pp["Expert position II"]
both = a.isin(cats) & b.isin(cats)
rows = []
for w in ["linear", "quadratic"]:
    rows.append((w, "(i) No opinion left out", int(both.sum()), vk.weighted_kappa(a[both], b[both], cats, w)))
    rows.append((w, "(ii) No opinion as Neutral", len(a), vk.weighted_kappa(a.replace("No opinion", "Neutral"), b.replace("No opinion", "Neutral"), cats, w)))
kap = pd.DataFrame(rows, columns=["weights", "No opinion", "pairs", "kappa"])
kap["within 0.84 +- 0.084"] = (kap["kappa"] - 0.84).abs() <= 0.084
print(kap.round(3).to_string(index=False))
print("unweighted agreement on the 266 pairs:", round(float((a[both] == b[both]).mean()), 3))
print(pd.crosstab(a, b).loc[cats + ["No opinion"], cats + ["No opinion"]])

**Result: reproduced, resting on an assumption.** With quadratic weights and pairs with "No opinion" left out (266 of 429 pairs), kappa is 0.842 (expected 0.84). The other readings give 0.734 (linear, left out), 0.620 (linear, as Neutral) and 0.745 (quadratic, as Neutral), all below the tolerance; the outcome rests on the quadratic weights and leaving "No opinion" out, which the paper does not state.

## uni-and-landscape-placements

> The platform placed each party on five uni-dimensions, using 18 of the 39 statements, and on a
> two-dimensional socio-economic and socio-cultural landscape from 0 to 100, using 16 statements; the
> placements (Figs. 2 and 3) conform to the anticipated placements of Slovak parties.

Assignments and polarities are those of the dataset's Supplementary Table 1. Scores are reading (a) of the
criterion; reading (b) (start at 50 and move by each statement's polarity-adjusted deviation, averaged) is
algebraically the same number, 50 + mean(p (x - 50)) = mean of x or 100 - x, so it is not computed apart.
"No opinion" is (i) left out or (ii) scored 50. Final positions are used (Methods: "the final authority on
party positioning rested solely with our Volebný Kompas team").

The expected placements below were read from the published Figures 2 and 3 of the version of record
(doi:10.1038/s41597-024-03777-0) by a separate agent that saw only the figure images, not the scores
computed here. It located each logo's centre in pixels and interpolated linearly between the axis ends
(Fig. 2: x = 30.5 is 0 and x = 1319 is 100 in the 1350-pixel image, checked by the "You" marker at 50.0;
Fig. 3: x = 51.5 to 897.5 and y = 54.5 to 901.5 for 0 to 100, top = liberal 0). In Fig. 2 the logos sit on
columns at whole percentages (0, 8, 16, 25, 33, 41, 50, ...: k times 100/12 cut to an integer), read to about
0.3; the foreign policy panel has finer positions. Fig. 3 is read to about 0.3 to 1 point (PS, clipped at the
top edge, and SME Rodina, partly hidden, to about 1). A party counts as matching its figure value within 1
point. Two parties are ordered differently when the figure and the scores order them opposite ways by more
than 1 point each; pairs within 1 point on either side count as ties.

In [ ]:
assign = vk.dimension_assignments(supp)
print(assign.groupby(["kind", "dimension"]).agg(statements=("statement", list), polarity=("polarity", list)).to_string())
print("statements on uni-dimensions:", assign.loc[assign.kind == "uni", "statement"].nunique(), "(expected 18)",
      "| on the landscape:", assign.loc[assign.kind == "landscape", "statement"].nunique(), "(expected 16)")

# Read from the published Fig. 2 and Fig. 3 by pixel position, by an agent that had not seen the scores (see the text above).
fig2 = {
    "ENVIRONMENT": {"ALIANCIA": 66, "DEMOKRATI": 33, "HLAS-SD": 41, "KDH": 33, "OLANO": 41, "PS": 83, "Republika": 8, "SME Rodina": 25, "SMER-SD": 41, "SNS": 0, "SaS": 25},
    "MULTILEVEL GOVERNANCE": {"ALIANCIA": 16, "DEMOKRATI": 16, "HLAS-SD": 16, "KDH": 25, "OLANO": 41, "PS": 41, "Republika": 33, "SME Rodina": 58, "SMER-SD": 58, "SNS": 50, "SaS": 25},
    "FORM OF GOVERNMENT": {"ALIANCIA": 50, "DEMOKRATI": 0, "HLAS-SD": 50, "KDH": 50, "OLANO": 50, "PS": 0, "Republika": 91, "SME Rodina": 66, "SMER-SD": 91, "SNS": 100, "SaS": 8},
    "EUROPEAN UNION": {"ALIANCIA": 66, "DEMOKRATI": 83, "HLAS-SD": 83, "KDH": 41, "OLANO": 66, "PS": 100, "Republika": 8, "SME Rodina": 33, "SMER-SD": 33, "SNS": 8, "SaS": 58},
    "FOREIGN POLICY": {"ALIANCIA": 70, "DEMOKRATI": 95, "HLAS-SD": 75, "KDH": 79, "OLANO": 83, "PS": 95, "Republika": 8, "SME Rodina": 75, "SMER-SD": 45, "SNS": 16, "SaS": 91},
}
fig3 = {"ECONOMIC DIMENSION": {"PS": 54.5, "SaS": 70.3, "DEMOKRATI": 52.2, "HLAS-SD": 27.1, "ALIANCIA": 38.5, "SMER-SD": 25.0, "SME Rodina": 47.6, "OLANO": 38.5, "SNS": 22.7, "Republika": 18.0, "KDH": 56.7},
        "CULTURAL DIMENSION": {"PS": 0.0, "SaS": 8.2, "DEMOKRATI": 33.1, "HLAS-SD": 45.6, "ALIANCIA": 50.0, "SMER-SD": 54.1, "SME Rodina": 58.4, "OLANO": 62.3, "SNS": 74.9, "Republika": 83.0, "KDH": 74.9}}
expected = pd.DataFrame({**fig2, **fig3})

positions = pp.pivot(index="Party", columns="statement", values="Final position")
num = positions.replace(vk.LIKERT).replace("No opinion", np.nan).astype(float)
scores = {"(i) No opinion left out": vk.dimension_scores(num, assign), "(ii) No opinion as 50": vk.dimension_scores(num.fillna(50), assign)}


def discordant(x, y, tol):
    # pairs of parties ordered opposite ways by the figure and the scores, each by more than the tolerance
    out = []
    idx = list(x.index)
    for i in range(len(idx)):
        for j in range(i + 1, len(idx)):
            p, q = idx[i], idx[j]
            dx, dy = x[p] - x[q], y[p] - y[q]
            if (dx > tol and dy < -tol) or (dx < -tol and dy > tol):
                out.append(f"{p}/{q}")
    return out


for name, sc in scores.items():
    print(f"\n{name}")
    res = []
    for d in expected.columns:
        x, y = expected[d], sc.loc[expected.index, d]
        diff = (y - x).abs()
        res.append((d, round(float(diff.max()), 1), diff.idxmax(), int((diff > 1).sum()), len(discordant(x, y, 1)), discordant(x, y, 1)[:6]))
    print(pd.DataFrame(res, columns=["dimension", "largest |score - figure|", "party", "parties off by more than 1", "pairs ordered differently", "examples"]).to_string(index=False))
econ = assign.dimension == "ECONOMIC DIMENSION"
print("\nEconomic axis, reading (ii), leaving out one statement of the economic dimension:")
print(pd.DataFrame({f"without statement {s}": vk.dimension_scores(num.fillna(50), assign[~(econ & (assign.statement == s))])["ECONOMIC DIMENSION"]
                    for s in [10, 21]}).assign(figure=expected["ECONOMIC DIMENSION"]).round(1).loc[["PS", "DEMOKRATI", "KDH", "SaS", "OLANO"]].to_string())
print("\nScores, reading (ii), with the figure values in brackets:")
s2 = scores["(ii) No opinion as 50"]
print(pd.DataFrame({d: [f"{s2.loc[p, d]:.1f} ({expected.loc[p, d]:.1f})" for p in expected.index] for d in expected.columns}, index=expected.index).to_string())

In [ ]:
s2 = scores["(ii) No opinion as 50"]
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(expected["ECONOMIC DIMENSION"], expected["CULTURAL DIMENSION"], marker="o", facecolors="none", edgecolors="k", label="Fig. 3, blind pixel reading")
ax.scatter(s2.loc[expected.index, "ECONOMIC DIMENSION"], s2.loc[expected.index, "CULTURAL DIMENSION"], marker="x", c="C3", label="computed (ii)")
for p in expected.index:
    ax.annotate(p, (s2.loc[p, "ECONOMIC DIMENSION"], s2.loc[p, "CULTURAL DIMENSION"]), fontsize=7)
ax.axvline(50, color="0.6", lw=0.8)
ax.axhline(50, color="0.6", lw=0.8)
ax.set_xlim(0, 100); ax.set_ylim(100, 0)
ax.set_xlabel("economic left (0) to right (100)"); ax.set_ylabel("liberal (0) to conservative (100)")
ax.legend(fontsize=7)
plt.show()
print("all computed points inside 0 to 100:", bool(((s2 >= 0) & (s2 <= 100)).all().all()))
print("PS, reading (ii): economic", round(float(s2.loc["PS", "ECONOMIC DIMENSION"]), 1), "| Fig. 3", fig3["ECONOMIC DIMENSION"]["PS"],
      "| PS's figure position in pixels from the centre line:", round((fig3["ECONOMIC DIMENSION"]["PS"] - 50) * (897.5 - 51.5) / 100, 1))


**Result: partially reproduced.** 18 and 16 statements are assigned, as stated. With "No opinion" as 50 (reading ii), every party's score on the five uni-dimensions is within 0.8 of the blind reading of Fig. 2 (the figure places them at whole percentages), and the cultural axis matches Fig. 3 for all 11 parties (largest difference 0.3), with no pair ordered differently. On the economic axis 10 parties are within 0.2 of Fig. 3, but PS scores 47.7 where the figure places it at 54.5, right of the centre line (38 pixels right of it in the published figure), so PS and DEMOKRATI (52.3; figure 52.2) are ordered differently. Leaving out statement 10 (on both axes) or 21 does not explain it: PS scores 52.5 or 47.5, still below DEMOKRATI (55.0 either way), and other parties move away from the figure. With "No opinion" left out (reading i) the placements differ from Fig. 2 on four of the five uni-dimensions. The order matches on six of the seven axes.

## party-clustering

> Hierarchical clustering of the parties' recoded positions, by Manhattan distance and average linkage,
> groups the parties largely as anticipated (Fig. 7).

The R code clusters `recoded_party_positions.csv`, the final positions with "No opinion" as 50 (README),
which is reading (a). Reading (b) leaves "No opinion" out of each pairwise distance (the distance is the mean
absolute difference over the statements both parties placed, times 39, so the scale is that of Fig. 7). The
expert-coded positions of each coder are a variant. The expected tree was read from the published Fig. 7 by
the same separate agent, blind to the computed tree: the clusters it forms, and each merge height from the
bar's pixel position against the axis ticks (1800 at y = 46.27, a linear fit of -4.180 per pixel), to about
+-3.

In [ ]:
def clusters(Z, labels):
    # every cluster of the tree as a set of labels, with its merge height
    sets = [frozenset([l]) for l in labels]
    out = {}
    for i, j, h, _ in Z:
        s = sets[int(i)] | sets[int(j)]
        sets.append(s)
        out[s] = h
    return out


fig7 = {frozenset(c.split(",")): h for c, h in [("Republika,SNS", 474), ("SaS,DEMOKRATI", 675), ("OLANO,KDH", 725), ("PS,SaS,DEMOKRATI", 787),
        ("SMER-SD,Republika,SNS", 887), ("HLAS-SD,ALIANCIA", 900), ("OLANO,KDH,HLAS-SD,ALIANCIA", 1100),
        ("SME Rodina,OLANO,KDH,HLAS-SD,ALIANCIA", 1181), ("PS,SaS,DEMOKRATI,SME Rodina,OLANO,KDH,HLAS-SD,ALIANCIA", 1315)]}
fig7_root = 1774


def compare(Z, labels, name):
    got = clusters(Z, labels)
    got = {k: v for k, v in got.items() if len(k) < len(labels)}
    same = [k for k in fig7 if k in got]
    print(f"{name}: {len(same)} of {len(fig7)} clusters of Fig. 7 found; root height {Z[-1, 2]:.1f}")
    for k in fig7:
        print(f"   {sorted(k)}: Fig. 7 {fig7[k]}, here {got.get(k, float('nan')):.1f}")
    extra = [sorted(k) for k in got if k not in fig7]
    print("   clusters here not in Fig. 7:", extra)
    return len(same)


labels = [n.replace("OĽANO", "OLANO") for n in rec.index]
Za = hierarchy.linkage(pdist(rec.to_numpy(float), "cityblock"), "average")
compare(Za, labels, "(a) recoded CSV, No opinion as 50")


def nan_cityblock(M):
    n = len(M)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            ok = ~np.isnan(M[i]) & ~np.isnan(M[j])
            D[i, j] = D[j, i] = np.abs(M[i, ok] - M[j, ok]).mean() * M.shape[1]
    return D[np.triu_indices(n, 1)]


fin = num.loc[[l for l in labels]]
compare(hierarchy.linkage(nan_cityblock(fin.to_numpy()), "average"), labels, "(b) final positions, No opinion left out")
for coder in ["Expert position I", "Expert position II"]:
    e = pp.pivot(index="Party", columns="statement", values=coder).replace(vk.LIKERT).replace("No opinion", 50).astype(float).loc[labels]
    compare(hierarchy.linkage(pdist(e.to_numpy(), "cityblock"), "average"), labels, f"variant: {coder}, No opinion as 50")
fig, ax = plt.subplots(figsize=(8, 3))
hierarchy.dendrogram(Za, labels=labels, ax=ax, leaf_rotation=90)
for h in list(fig7.values()) + [fig7_root]:
    ax.axhline(h, color="0.6", lw=0.6, ls="--")
ax.set_ylabel("Manhattan distance")
ax.set_title("Reading (a): recoded positions, average linkage; dashed: merge heights read from Fig. 7", fontsize=9)
plt.show()
print("root height: computed", round(float(Za[-1, 2]), 1), "| Fig. 7", fig7_root)

**Result: reproduced.** With the recoded positions (reading a) all 9 clusters of Fig. 7 below the root are found and no other, at heights within 3 of those read from the figure (root 1774.0, figure 1774). Leaving "No opinion" out (reading b), or using either coder's positions, changes the tree.

## table2-gender, table2-age, table2-education

> Of the responses counted in Table 2, 41.61% are female (44,919), 57.08% male (61,626) and 1.32% other or
> prefer not to say (1,420). / Respondents are skewed toward younger ages ... / toward the well educated ...

Readings from the criterion: (a) cleaned rows with the variable given; (b) raw rows with it given; (c) cleaned
rows with all three given; (d) raw rows with all three given. Gender, age and education are empty in the
same rows (data check), so (a) equals (c) and (b) equals (d). "Cleaned" is the paper's cleaning; the R code's
cleaning, which also drops the 330 straight-line rows, is shown as a variant (e).

In [ ]:
table2 = {
    "GENDER": {"Female": 44919, "Male": 61626, "Other / Prefer not to say": 1420},
    "AGE": {"Under 18": 1264, "18-20": 7020, "21-24": 12280, "25-34": 34894, "35-44": 27847, "45-54": 15178, "55-64": 6083, "65-74": 2716, "75 or older": 683},
    "EDUCATION": {"No education": 990, "Primary education": 1665, "Secondary education": 30644, "Applied higher education": 2570,
                  "Vocational education": 2554, "University degree/Academic higher education": 69542},
}
table2_pct = {"Female": 41.61, "Male": 57.08, "Other / Prefer not to say": 1.32, "Under 18": 1.17, "18-20": 6.50, "21-24": 11.38, "25-34": 32.32, "35-44": 25.79,
              "45-54": 14.06, "55-64": 5.63, "65-74": 2.52, "75 or older": 0.63, "No education": 0.92, "Primary education": 1.54, "Secondary education": 28.38,
              "Applied higher education": 2.38, "Vocational education": 2.37, "University degree/Academic higher education": 64.41}
all3 = raw[["GENDER", "AGE", "EDUCATION"]].ne("").all(axis=1)
row_readings = {"(a) cleaned, variable given": ("paper", False), "(b) raw, variable given": ("raw", False),
                "(c) cleaned, all three given": ("paper", True), "(d) raw, all three given": ("raw", True),
                "(e) R code's cleaning, variable given": ("code", False)}
t2 = {}
for var, cats_ in table2.items():
    out = pd.DataFrame({"paper n": pd.Series(cats_), "paper %": pd.Series({c: table2_pct[c] for c in cats_})})
    for name, (s, need_all) in row_readings.items():
        m = sel[s] & (all3 if need_all else raw[var].ne(""))
        n = raw.loc[m, var].value_counts().reindex(list(cats_)).fillna(0).astype(int)
        out[name + " n"] = n
        out[name + " %"] = (100 * n / n.sum()).round(2)
    t2[var] = out
    print(f"\n{var}: totals", {k: int(out[k + ' n'].sum()) for k in row_readings}, "| paper total", int(out["paper n"].sum()))
    print(out.to_string())

In [ ]:
checks = {"table2-gender": ("GENDER", list(table2["GENDER"]), list(table2["GENDER"])),
          "table2-age": ("AGE", list(table2["AGE"]), ["25-34", "35-44"]),
          "table2-education": ("EDUCATION", list(table2["EDUCATION"]), ["University degree/Academic higher education"])}
for claim, (var, count_cats, pct_cats) in checks.items():
    out = t2[var]
    for name in row_readings:
        n_ok = ((out.loc[count_cats, name + " n"] - out.loc[count_cats, "paper n"]).abs() <= 0.1 * out.loc[count_cats, "paper n"] + 1e-9)
        p_ok = ((out.loc[pct_cats, name + " %"] - out.loc[pct_cats, "paper %"]).abs() <= 0.1 * out.loc[pct_cats, "paper %"] + 1e-9)
        largest = ((out.loc[count_cats, name + " n"] - out.loc[count_cats, "paper n"]).abs() / out.loc[count_cats, "paper n"]).max()
        print(f"{claim} {name}: counts within 10% {int(n_ok.sum())}/{len(n_ok)}, percentages {int(p_ok.sum())}/{len(p_ok)}; largest relative count difference {largest:.2%}")

**Result: table2-gender reproduced.** Under reading (a) (cleaned rows, gender given) the counts are 44,938 female, 61,670 male and 1,455 other (41.59%, 57.07%, 1.35%), within tolerance; every reading is. The R code's cleaning gives Table 2 exactly (44,919, 61,626 and 1,420 of 107,965).

**Result: table2-age reproduced.** Under reading (a) every age count is within 3% of Table 2 (25-34: 34,914 against 34,894; 35-44: 27,870 against 27,847) and the two percentages match; the R code's cleaning gives every count of Table 2 exactly.

**Result: table2-education reproduced.** Under reading (a) every education count is within 3% of Table 2 (university 69,575, 64.38%, against 69,542, 64.41%; no education 1,015 against 990); the R code's cleaning gives every count exactly.

## age-education-inconsistencies

> The multivariate check found 40 respondents under 18 and 400 aged 18 to 20 reporting university-level
> education, which is not congruent with their age.

Readings: rows (a) cleaned (paper's cleaning), (b) raw, and the R code's cleaning as a variant; education (i)
"University degree/Academic higher education" only, (ii) with "Applied higher education" too. The R code's
`validator` rules (printed in "The deposited R code") check only that AGE and EDUCATION take one of their
categories; no rule relates the two, so the code does not contain this check.

In [ ]:
uni = ["University degree/Academic higher education"]
rows = []
for rname, s in [("(a) cleaned", "paper"), ("(b) raw", "raw"), ("variant: R code's cleaning", "code")]:
    for ename, eds in [("(i) university", uni), ("(ii) university or applied higher", uni + ["Applied higher education"])]:
        m = sel[s] & raw["EDUCATION"].isin(eds)
        u18 = int((m & (raw["AGE"] == "Under 18")).sum())
        a1820 = int((m & (raw["AGE"] == "18-20")).sum())
        rows.append((rname, ename, u18, a1820, abs(u18 - 40) <= 4 and abs(a1820 - 400) <= 40))
print(pd.DataFrame(rows, columns=["rows", "education", "under 18", "18-20", "both within 10%"]).to_string(index=False))

**Result: partially reproduced.** One of the two counts is within tolerance. With the university category only, 378 respondents aged 18-20 on the cleaned rows (a, i) and 382 on the raw rows are within 40 of 400, but 27 are under 18 on both, not within 4 of 40. With applied higher education added, the counts are 34 and 505 (cleaned) and 34 and 509 (raw), so neither is within tolerance. No reading gives the under-18 count. The R code's rules do not relate age and education.

## s2-s3-wide-range and s4-s5-consensus

> S2 and S3 ... means of 43 and 42, standard deviations of 33 and medians of 25.
> S4 and S5 ... medians of 75 (means 70 and 69, standard deviations 26 and 29), though more than 17,435
> respondents for S4 and 23,344 for S5 disagree with scores of 0 or 25.

"No opinion" is left out (the paper: "prior to the re-coding of 'no opinion' with a value of 50"); rows (a) the
paper's cleaning, (b) raw rows with -1 left out of each statement; the R code's cleaning is a variant. Standard
deviation with n - 1.

In [ ]:
expect = {"S2": (43, 33, 25), "S3": (42, 33, 25), "S4": (70, 26, 75), "S5": (69, 29, 75)}
counts_paper = {"S4": 17435, "S5": 23344}
rows = []
for rname, s in [("(a) cleaned", "paper"), ("(b) raw", "raw"), ("variant: R code's cleaning", "code")]:
    for st, (m, sd, med) in expect.items():
        x = likert.loc[sel[s], st]
        x = x[x.notna() & (x != -1)]
        low = int(x.isin([0, 25]).sum())
        ok = abs(x.mean() - m) <= 0.1 * m and abs(x.std(ddof=1) - sd) <= 0.1 * sd and abs(x.median() - med) <= 0.1 * med
        rows.append((rname, st, len(x), round(x.mean(), 2), round(x.std(ddof=1), 2), x.median(), low, ok,
                     (abs(low - counts_paper[st]) <= 0.1 * counts_paper[st]) if st in counts_paper else None,
                     (low > counts_paper[st]) if st in counts_paper else None))
stats_t = pd.DataFrame(rows, columns=["rows", "statement", "n", "mean", "sd", "median", "n at 0 or 25", "mean, sd, median within 10%",
                                      "(i) count within 10%", "(ii) count above paper"])
print(stats_t.to_string(index=False))

**Result: s2-s3-wide-range reproduced.** On the cleaned rows S2 has mean 43.77, s.d. 33.82 and median 25, and S3 mean 42.09, s.d. 33.69 and median 25, all within tolerance. On the raw rows the S2 median is 50, outside it.

**Result: s4-s5-consensus reproduced.** On the cleaned rows S4 has mean 70.36, s.d. 26.58, median 75 and 17,460 answers of 0 or 25 (paper 17,435), and S5 mean 69.38, s.d. 29.53, median 75 and 23,369 (paper 23,344); both counts are within tolerance (reading i) and above the paper's numbers (reading ii). The R code's cleaning gives the paper's counts exactly (17,435 and 23,344), so "more than" reads as "as many as".

## efa-eleven-factors

> Parallel analyses on both the polychoric and the Pearson correlation matrices of S1 to S39 indicated that 11
> factors should be extracted.

Readings of the rows, from the criterion: (a) cleaned rows, "no opinion" as 50; (b) cleaned rows, "no opinion"
missing, pairwise complete. Settings, from the criterion: psych's fa.parallel defaults (minres, 20 iterations,
the factor count). The deposited R code fixes settings the paper does not state, taken here as a variant
(author, step 1.6): rows of the R code's cleaning whose EMAIL_ID occurs once ("FILTERING DATA WITH UNIQUE
EMAIL ID"), "no opinion" as 50, `fa.parallel(..., fm = "ml", fa = "fa", n.iter = 1000)` for Pearson and
`n.iter = 100` for polychoric. Reversing statements does not change eigenvalues, so it is not applied.
The reference eigenvalues come from normal random data with the same number of rows (for the R code's
polychoric setting, the first 100 of the 1,000 ML iterations); the generator is seeded
(a standard choice), and the reference's spread over iterations is printed to show how much the seed can move
the count.

A short synthetic check of the factor routines comes first: with a known 3-factor population correlation
matrix the communalities should be recovered and parallel analysis on 2,000 simulated rows should give 3.

In [ ]:
rng = np.random.default_rng(20240709)
Lt = np.zeros((12, 3))
Lt[np.arange(12), np.arange(12) % 3] = rng.uniform(0.5, 0.8, 12)
Lt[np.arange(12), (np.arange(12) + 1) % 3] = rng.uniform(0.0, 0.3, 12)
Rt = Lt @ Lt.T
np.fill_diagonal(Rt, 1)
for fm in ["ml", "pa"]:
    Lh = vk.fa_loadings(Rt, 3, fm)
    print(fm, "largest communality error:", float(np.abs((Lh ** 2).sum(1) - (Lt ** 2).sum(1)).max().round(4)))
Zs = rng.multivariate_normal(np.zeros(12), Rt, size=2000)
Rs = np.corrcoef(Zs, rowvar=False)
print("parallel analysis on simulated data:", vk.parallel_count(vk.reduced_eigenvalues(Rs, "minres"), vk.parallel_reference(2000, 12, "minres", 20, rng)), "factors (true 3)")

In [ ]:
import time
n_fits_synthetic = len(vk.FITS)
rng = np.random.default_rng(39)
t0 = time.time()
mats = {}
Xa = likert.loc[sel["paper"], vk.STATEMENTS].fillna(50)
Xb = likert.loc[sel["paper"], vk.STATEMENTS]
Xc = likert.loc[sel["code_email"], vk.STATEMENTS].fillna(50)
for name, X in [("(a) cleaned, no opinion 50", Xa), ("(b) cleaned, no opinion missing", Xb), ("variant: R code's unique-email rows", Xc)]:
    mats[name] = {"n": len(X), "Pearson": vk.pearson(X), "polychoric": vk.polychoric(X)}
    print(f"{name}: {len(X)} rows; smallest eigenvalue Pearson {np.linalg.eigvalsh(mats[name]['Pearson']).min():.3f}, "
          f"polychoric {np.linalg.eigvalsh(mats[name]['polychoric']).min():.3f}; {time.time() - t0:.0f} s")
print("largest |polychoric - Pearson|, reading (a):", float(np.abs(mats["(a) cleaned, no opinion 50"]["polychoric"] - mats["(a) cleaned, no opinion 50"]["Pearson"]).max().round(3)))

In [ ]:
refs = {}
n0 = len(vk.FITS)
for n in sorted({m["n"] for m in mats.values()}):
    t0 = time.time()
    sims = {}
    for fm, iters in [("minres", 20), ("ml", 1000)]:
        vals = []
        for _ in range(iters):
            Z = rng.standard_normal((n, 39), dtype=np.float32).astype(np.float64)
            vals.append(vk.reduced_eigenvalues(np.corrcoef(Z, rowvar=False), fm))
        sims[fm] = np.array(vals)
    refs[n] = sims
    print(f"n = {n}: references in {time.time() - t0:.0f} s")
ref_fits = vk.FITS[n0:]
obs_fits = []
rows = []
for name, m in mats.items():
    for cor in ["Pearson", "polychoric"]:
        for setting, fm, iters in [("defaults (minres, 20)", "minres", 20), ("R code (ml, %d)" % (1000 if cor == "Pearson" else 100), "ml", 1000 if cor == "Pearson" else 100)]:
            n1 = len(vk.FITS)
            obs = vk.reduced_eigenvalues(m[cor], fm)
            obs_fits += vk.FITS[n1:]
            sim = refs[m["n"]][fm][:iters]
            k = vk.parallel_count(obs, sim.mean(0))
            k95 = vk.parallel_count(obs, np.quantile(sim, 0.95, axis=0))
            rows.append((name, cor, setting, k, k95, round(obs[k - 1], 4), round(sim.mean(0)[k - 1], 4), round(obs[k], 4), round(sim.mean(0)[k], 4)))
pa = pd.DataFrame(rows, columns=["rows", "correlation", "settings", "factors", "factors (95th percentile)", "last value above", "its reference", "next value", "its reference"])
print(pa.to_string(index=False))
print("uniqueness fits that did not converge: synthetic check", vk.FITS[:n_fits_synthetic].count(False), "of", n_fits_synthetic,
      "| reference (simulated) matrices", ref_fits.count(False), "of", len(ref_fits), "| observed matrices", obs_fits.count(False), "of", len(obs_fits))

psych's `fa.parallel` with `cor = "poly"` on raw data, the claims file's default reading, does not compare
with simulated normal data: it resamples each column on its own (which keeps the margins and removes the
correlations) and takes polychoric correlations of the resampled data. The cell below computes that reference
for readings (a) and (b), with minres and 20 iterations, from its own seeded generator. The R code's polychoric
call passes a correlation matrix with `n.obs`, for which psych can only simulate normal data, as above.

In [ ]:
rng_poly = np.random.default_rng(4139)
rows = []
for name, X in [("(a) cleaned, no opinion 50", Xa), ("(b) cleaned, no opinion missing", Xb)]:
    t0 = time.time()
    n0 = len(vk.FITS)
    sim = vk.parallel_reference_resampled(X, "minres", 20, rng_poly)
    n_ref = len(vk.FITS) - n0
    bad_ref = vk.FITS[n0:].count(False)
    obs = vk.reduced_eigenvalues(mats[name]["polychoric"], "minres")
    k = vk.parallel_count(obs, sim.mean(0))
    k95 = vk.parallel_count(obs, np.quantile(sim, 0.95, axis=0))
    rows.append((name, k, k95, round(obs[k - 1], 4), round(sim.mean(0)[k - 1], 4), round(obs[k], 4), round(sim.mean(0)[k], 4),
                 round(sim[:, k].std(ddof=1), 4), f"{bad_ref} of {n_ref}", round(time.time() - t0)))
print(pd.DataFrame(rows, columns=["rows", "factors", "factors (95th percentile)", "last value above", "its reference", "next value",
                                  "its reference", "sd of that reference over iterations", "reference fits not converged", "seconds"]).to_string(index=False))


**Result: reproduced.** On the cleaned rows with "no opinion" as 50 (reading a), parallel analysis with psych's defaults indicates 11 factors for both the Pearson and the polychoric matrix; the 11th Pearson factor value is above its reference (0.030 against 0.013) and the 12th below it (0.004 against 0.012). For the polychoric matrix the 12th value, 0.0106, is close to the normal-data reference (0.0117), but against psych's own reference for cor = "poly", polychoric correlations of resampled data, it is further below (0.0143, s.d. over iterations 0.0009), and the count is 11 again. With "no opinion" missing (reading b) the default counts are 12 (Pearson) and 11 (polychoric, against either reference), so the claim holds under reading (a) only with the defaults. With the R code's settings (ML, 1000 or 100 iterations) every row reading gives 11 and 11, and so do the code's unique-email rows; the 95th percentile of the reference gives the same counts as its mean. The 18 uniqueness fits that did not converge are all on simulated reference matrices (18 of 2,040); every fit on an observed matrix converged (12 of 12), and so did every fit on resampled data (40 of 40).

## efa-loadings-consistent

> With 11 factors and promax rotation, maximum likelihood and principal axis factoring gave consistent results,
> and S1 (real estate tax) and S3 (retirement age) loaded strongly.

From the criterion: ML and principal axis (psych fa, rotate "promax"), 11 factors, on the matrices and rows of
efa-eleven-factors. S1 and S3 load strongly when their largest absolute loading is at least 0.4 (reading i) or
0.5 (reading ii) in both solutions; the solutions are consistent when, with factors matched by Tucker congruence,
at least 35 of 39 statements have their largest loading on matched factors. These thresholds are the claims
file's own, so an outcome rests on them.

In [ ]:
rows = []
loadings = {}
for name, m in mats.items():
    for cor in ["Pearson", "polychoric"]:
        L_ml = vk.promax(vk.fa_loadings(m[cor], 11, "ml"))
        L_pa = vk.promax(vk.fa_loadings(m[cor], 11, "pa"))
        loadings[(name, cor)] = (L_ml, L_pa)
        C = vk.tucker_congruence(L_ml, L_pa)
        r, c = linear_sum_assignment(-np.abs(C))
        match = dict(zip(r, c))
        agree = int(sum(match[i] == j for i, j in zip(np.abs(L_ml).argmax(1), np.abs(L_pa).argmax(1))))
        s1 = (np.abs(L_ml[0]).max(), np.abs(L_pa[0]).max())
        s3 = (np.abs(L_ml[2]).max(), np.abs(L_pa[2]).max())
        rows.append((name, cor, round(s1[0], 3), round(s1[1], 3), round(s3[0], 3), round(s3[1], 3), agree,
                     round(float(np.abs(C[r, c]).min()), 3), min(s1 + s3) >= 0.4 and agree >= 35, min(s1 + s3) >= 0.5 and agree >= 35))
ef = pd.DataFrame(rows, columns=["rows", "correlation", "S1 ML", "S1 PA", "S3 ML", "S3 PA", "statements on matched factors (of 39)",
                                 "lowest congruence", "holds (i) 0.4", "holds (ii) 0.5"])
print(ef.to_string(index=False))
print("uniqueness fits that did not converge:", vk.FITS.count(False), "of", len(vk.FITS))
L_ml, L_pa = loadings[("(a) cleaned, no opinion 50", "Pearson")]
print("\nPromax loadings, reading (a), Pearson, ML (|loading| >= 0.3 shown):")
print(pd.DataFrame(np.where(np.abs(L_ml) >= 0.3, L_ml, np.nan), index=vk.STATEMENTS, columns=[f"F{i + 1}" for i in range(11)]).round(2).to_string(na_rep=""))

**Result: partially reproduced, resting on an assumption.** On the Pearson matrix of the cleaned rows (reading a), S1 loads 0.523 (ML) and 0.502 (PA) and S3 0.598 and 0.603, and 35 of 39 statements fall on matched factors (lowest congruence 0.892), so the claim holds there at both thresholds; reading (b) gives the same (S1 0.553 and 0.539, 35 of 39). On the polychoric matrices only 32 of 39 statements match (lowest congruence 0.355 and 0.358), so the two solutions are not consistent by the claims file's rule. The criterion takes the matrices of efa-eleven-factors, both Pearson and polychoric, so the claim holds on one of the two. The R code runs ML and PA only on the Pearson matrix; on its unique-email rows S1 loads 0.379 in the PA solution, with 33 of 39 matched. The outcome rests on the claims file's thresholds, which the paper does not give.

## Summary

| Claim | Outcome |
|---|---|
| user-responses-count | reproduced |
| collection-window | partially reproduced |
| party-positions-structure | reproduced |
| eleven-parties | reproduced |
| party-self-placements | reproduced |
| coder-agreement-kappa | reproduced (assumed: quadratic weights, No opinion left out) |
| uni-and-landscape-placements | partially reproduced |
| party-clustering | reproduced |
| table2-gender, table2-age, table2-education | reproduced |
| age-education-inconsistencies | partially reproduced |
| s2-s3-wide-range, s4-s5-consensus | reproduced |
| efa-eleven-factors | reproduced |
| efa-loadings-consistent | partially reproduced (assumed: the claims file's thresholds) |
| selected-parties-top-vote | blocked |
| website-views | blocked |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's: it is in Python, not R; the parallel analysis and the factor
extraction are reimplemented after psych (see `vk.py`), and the polychoric correlations use thresholds from each
statement's margins without psych's correction for empty cells. Figures 2, 3 and 7 were compared by values read
from the published figures by pixel position, by an agent that had not seen the computed values.

What was found about the dataset: the party workbook numbers statements 16 to 24 wrongly ("Statement no." does
not match the wording in 99 rows); the R code removes 330 straight-line responses that the paper's cleaning does
not mention, and with that step Table 2 and the S4/S5 counts are reproduced exactly; of the age-education counts of the structural validation (40 and 400) only the second is reproduced, and the R code holds no such check; 432 responses are dated after
election day, including the sample row of Table 1; "no opinion" is `NA` in the file, not an empty string; the
self-placement source column is empty; Fig. 3 places PS right of centre on the economic axis, where the deposited
final positions give 47.7.